In [1]:
import sys; sys.path.append("../src")
import pandas as pd, time
from common import clean_name, core_name, clean_addr, addr_numbers

s1 = pd.read_pickle("../work/tr_s1.pkl")
s2 = pd.read_pickle("../work/tr_s2.pkl")
s3 = pd.read_pickle("../work/tr_s3.pkl")

sample = pd.concat([s1.head(6), s2.head(8), s3.head(8)])
for _, r in sample.iterrows():
    cn = clean_name(r.business_name)
    ca = clean_addr(r.business_address, r.country)
    print(r.entity_id, "|", r.country)
    print("   NAME :", r.business_name, " -> ", cn, " || core:", core_name(cn))
    print("   ADDR :", r.business_address, " -> ", ca, " || nums:", addr_numbers(ca))

t = time.time()
tmp = s2.head(100000)
_ = [clean_name(x) for x in tmp.business_name]
_ = [clean_addr(a, c) for a, c in zip(tmp.business_address, tmp.country)]
print("\nSeconds for 100k rows:", round(time.time() - t, 1))


S1-925783039 | US
   NAME : Orelee's Barbershop  ->  orelee s barbershop  || core: orelee s barbershop
   ADDR : 1795 Westchester Drive, High Point, NC  ->  1795 westchester drive high point nc  || nums: 1795
S1-773889195 | US
   NAME : Prime Money  ->  prime money  || core: prime money
   ADDR : 17560 Ellis Road, Tahlequah, OK  ->  17560 ellis road tahlequah ok  || nums: 17560
S1-377745466 | US
   NAME : B+ Retail Inc  ->  b retail incorporated  || core: b retail
   ADDR : 1712 Montebello Avenue, Phoenix, AZ  ->  1712 montebello avenue phoenix az  || nums: 1712
S1-133037285 | US
   NAME : Christ Chapel  ->  christ chapel  || core: christ chapel
   ADDR : 2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD  ->  2100 cameron drive unit apartment g dundalk md  || nums: 2100
S1-755362802 | India
   NAME : Prabhav Business Center  ->  prabhav business center  || core: prabhav business center
   ADDR : 797, Lake Town Block A, Kolkata, Howrah, West Bengal  ->  797 lake town block a kolkata how

In [1]:
import sys; sys.path.append("../src")
import pandas as pd, time, gc
from common import clean_df

for name in ["tr_s1", "tr_s2", "tr_s3", "te_s1", "te_s2", "te_s3"]:
    t = time.time()
    df = clean_df(pd.read_pickle(f"../work/{name}.pkl"))
    df.to_pickle(f"../work/clean_{name}.pkl")
    print(name, len(df), "rows,", round(time.time() - t), "s")
    del df; gc.collect()

tr_s1 2206821 rows, 43 s
tr_s2 5034616 rows, 113 s
tr_s3 5285603 rows, 121 s
te_s1 1732544 rows, 36 s
te_s2 4887273 rows, 113 s
te_s3 5082316 rows, 237 s


In [2]:
%pip install scikit-learn sparse_dot_topn rapidfuzz lightgbm anyascii

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Rehan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, time, os
from blocking import generate_candidates_big

cols = ["entity_id", "country", "core", "addr"]
s1 = pd.read_pickle("../work/clean_tr_s1.pkl")[cols]
other = pd.concat([pd.read_pickle("../work/clean_tr_s2.pkl")[cols],
                   pd.read_pickle("../work/clean_tr_s3.pkl")[cols]], ignore_index=True)
gt = pd.read_pickle("../work/tr_gt.pkl")

# Answer key as (S1, matched record) pairs
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
pairs = g[["source1_entity_id", "mid"]].explode("mid")

# 300k S1 businesses -> their records + a realistic share of 'belongs to nobody' records
s1_sample = s1.sample(300000, random_state=11).entity_id
owned = pairs[pairs.source1_entity_id.isin(set(s1_sample))].mid
unowned = other.entity_id[~other.entity_id.isin(set(pairs.mid))]
q = pd.concat([other[other.entity_id.isin(set(owned))],
               other[other.entity_id.isin(set(unowned.sample(int(len(owned) * 0.35), random_state=11)))]])
print("S1 (full):", len(s1), "| records to match:", len(q))

os.makedirs("../work/cand_train", exist_ok=True)
pd.Series(s1_sample.values).to_pickle("../work/cand_train/s1_sample_ids.pkl")
t = time.time()
files = generate_candidates_big(s1, q, "../work/cand_train/part")
print("DONE in", round((time.time() - t) / 60, 1), "minutes")

# Recall check: how many true matches made it into the shortlist
cand = pd.concat([pd.read_pickle(f) for f in files])
tp = pairs[pairs.source1_entity_id.isin(set(s1_sample))].rename(
    columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
print("Pairs:", len(cand), "| Recall:", round(len(cand.merge(tp, on=["entity_id", "s1_id"])) / len(tp), 3))

S1 (full): 2206821 | records to match: 1402581
India rows 0-500,000: 9,579,733 pairs, 7224s
India rows 500,000-561,727: 1,212,090 pairs, 660s
US rows 0-500,000: 9,401,644 pairs, 5301s
US rows 500,000-840,854: 6,535,678 pairs, 3911s
DONE in 286.9 minutes
Pairs: 26729145 | Recall: 0.967


In [1]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, glob, time
import lightgbm as lgb
from features import build_features
from evaluate import load_truth, split_ids, tune_threshold

cols = ["entity_id", "country", "core", "addr", "nums"]
gt = pd.read_pickle("../work/tr_gt.pkl")
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
pairs = g[["source1_entity_id", "mid"]].explode("mid")
owner = dict(zip(pairs.mid, pairs.source1_entity_id))

# 1. Load the realistic shortlist and keep records of 60k S1 businesses (+ some 'nobody' records)
cand = pd.concat([pd.read_pickle(f) for f in glob.glob("../work/cand_train/part_*.pkl")], ignore_index=True)
s1_sample = pd.read_pickle("../work/cand_train/s1_sample_ids.pkl")
sub = set(s1_sample.sample(60000, random_state=1))
qids = cand.entity_id.unique()
owned_ids = [e for e in qids if owner.get(e) in sub]
unowned_ids = [e for e in qids if e not in owner]
rng = np.random.default_rng(1)
keep = set(owned_ids) | set(rng.choice(unowned_ids, size=int(len(owned_ids) * 0.35), replace=False))
cand = cand[cand.entity_id.isin(keep)].reset_index(drop=True)
cand = cand.drop(columns=["name_sim", "addr_sim", "name_rank", "addr_rank"])
print("Pairs used:", len(cand))

# 2. Cleaned text for both sides
qc = pd.concat([pd.read_pickle("../work/clean_tr_s2.pkl")[cols],
                pd.read_pickle("../work/clean_tr_s3.pkl")[cols]])
qc = qc[qc.entity_id.isin(keep)]
s1c = pd.read_pickle("../work/clean_tr_s1.pkl")[cols]
s1c = s1c[s1c.entity_id.isin(set(cand.s1_id))]

# 3. Features + labels
t = time.time()
feat, FEATS = build_features(cand, s1c, qc)
print("Features time (s):", round(time.time() - t))
lab = pairs.rename(columns={"mid": "entity_id", "source1_entity_id": "s1_id"}).assign(label=1)
feat = feat.merge(lab, on=["entity_id", "s1_id"], how="left")
feat["label"] = feat["label"].fillna(0).astype(int)

# 4. Practice split, train, score
truth_all = load_truth(gt)
tr_ids, val_ids = split_ids(truth_all.keys())
q_val = {e for e in keep if owner.get(e) in val_ids or (e not in owner and rng.random() < 0.2)}
is_val = feat.entity_id.isin(q_val)

model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           n_jobs=-1, verbose=-1)
t = time.time()
model.fit(feat.loc[~is_val, FEATS], feat.loc[~is_val, "label"])
print("Training time (s):", round(time.time() - t))

pv = feat.loc[is_val, ["entity_id", "s1_id"]].rename(columns={"entity_id": "other_id"})
pv["prob"] = model.predict_proba(feat.loc[is_val, FEATS])[:, 1]
eval_ids = [s for s in sub if s in val_ids]
best_t, best_s = tune_threshold(pv, truth_all, eval_ids)
print(pd.Series(model.feature_importances_, FEATS).sort_values(ascending=False).head(10))

import os
os.makedirs("../models", exist_ok=True)
tune_threshold(pv, truth_all, eval_ids, thresholds=[0.05, 0.10, 0.15, 0.20, 0.30])
model.booster_.save_model("../models/model_v1.txt")
print("Model saved")

Pairs used: 5329543
Features time (s): 267
Training time (s): 62
threshold 0.30 -> F0.5 0.9672
threshold 0.35 -> F0.5 0.9648
threshold 0.40 -> F0.5 0.9611
threshold 0.45 -> F0.5 0.9557
threshold 0.50 -> F0.5 0.9494
threshold 0.55 -> F0.5 0.9460
threshold 0.60 -> F0.5 0.9431
threshold 0.65 -> F0.5 0.9409
threshold 0.70 -> F0.5 0.9377
threshold 0.75 -> F0.5 0.9331
threshold 0.80 -> F0.5 0.9287
threshold 0.85 -> F0.5 0.9233
threshold 0.90 -> F0.5 0.9133
threshold 0.95 -> F0.5 0.8879
BEST: 0.3 0.9672
addr_sort        1931
name_jw          1913
name_partial     1833
name_nospace     1737
addr_jac         1707
name_jw_gap      1590
addr_set         1491
name_sort        1408
name_len_diff    1405
name_set         1286
dtype: int32
threshold 0.05 -> F0.5 0.9807
threshold 0.10 -> F0.5 0.9792
threshold 0.15 -> F0.5 0.9777
threshold 0.20 -> F0.5 0.9757
threshold 0.30 -> F0.5 0.9672
BEST: 0.05 0.9807
Model saved


In [2]:
from evaluate import f05_score, pairs_to_pred

# 1. Try lower thresholds
tune_threshold(pv, truth_all, eval_ids, thresholds=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30])

# 2. Ceiling: a 'perfect' model that knows which shortlisted pairs are true
perfect = feat.loc[is_val & (feat.label == 1), ["entity_id", "s1_id"]].rename(columns={"entity_id": "other_id"})
perfect["prob"] = 1.0
print("\nCeiling (perfect model, current shortlist):", round(f05_score(pairs_to_pred(perfect, 0.5), truth_all, eval_ids), 4))

threshold 0.05 -> F0.5 0.9804
threshold 0.10 -> F0.5 0.9793
threshold 0.15 -> F0.5 0.9777
threshold 0.20 -> F0.5 0.9764
threshold 0.25 -> F0.5 0.9748
threshold 0.30 -> F0.5 0.9729
BEST: 0.05 0.9804

Ceiling (perfect model, current shortlist): 0.9876


In [3]:
rng2 = np.random.default_rng(3)
half = set(rng2.choice(eval_ids, size=len(eval_ids) // 2, replace=False))
owner_of = pv.other_id.map(owner)
pv_sim = pv[owner_of.isin(half) | owner_of.isna()]
print("Share of 'nobody' records:", round(owner_of[pv_sim.index].isna().groupby(pv_sim.other_id).first().mean(), 3))
tune_threshold(pv_sim, truth_all, list(half), thresholds=[0.05, 0.10, 0.20, 0.30, 0.40, 0.50])

Share of 'nobody' records: 0.408
threshold 0.05 -> F0.5 0.9788
threshold 0.10 -> F0.5 0.9779
threshold 0.20 -> F0.5 0.9750
threshold 0.30 -> F0.5 0.9713
threshold 0.40 -> F0.5 0.9664
threshold 0.50 -> F0.5 0.9587
BEST: 0.05 0.9788


(0.05, 0.9788230914952456)

In [2]:
COUNTRY = "US"

import sys; sys.path.append("../src")
import pandas as pd, os, time
from blocking import generate_candidates_fast

cols = ["entity_id", "country", "core", "addr"]
s1 = pd.read_pickle("../work/clean_te_s1.pkl")[cols]
q = pd.concat([pd.read_pickle("../work/clean_te_s2.pkl")[cols],
               pd.read_pickle("../work/clean_te_s3.pkl")[cols]], ignore_index=True)
s1, q = s1[s1.country == COUNTRY], q[q.country == COUNTRY]
print(COUNTRY, "| S1:", len(s1), "| records:", len(q))

os.makedirs("../work/cand_test_fast", exist_ok=True)
t = time.time()
files = generate_candidates_fast(s1, q, "../work/cand_test_fast/part")
print("DONE", COUNTRY, "in", round((time.time() - t) / 60, 1), "min")

US | S1: 663106 | records: 3817031
US rows 0-250,000 of 3,817,031: 6,254,881 pairs, 103s
US rows 250,000-500,000 of 3,817,031: 6,250,357 pairs, 105s
US rows 500,000-750,000 of 3,817,031: 6,250,233 pairs, 102s
US rows 750,000-1,000,000 of 3,817,031: 6,251,840 pairs, 103s
US rows 1,000,000-1,250,000 of 3,817,031: 6,248,418 pairs, 103s
US rows 1,250,000-1,500,000 of 3,817,031: 6,249,167 pairs, 102s
US rows 1,500,000-1,750,000 of 3,817,031: 6,249,859 pairs, 101s
US rows 1,750,000-2,000,000 of 3,817,031: 6,259,267 pairs, 99s
US rows 2,000,000-2,250,000 of 3,817,031: 6,269,587 pairs, 98s
US rows 2,250,000-2,500,000 of 3,817,031: 6,271,990 pairs, 99s
US rows 2,500,000-2,750,000 of 3,817,031: 6,270,018 pairs, 101s
US rows 2,750,000-3,000,000 of 3,817,031: 6,271,810 pairs, 90s
US rows 3,000,000-3,250,000 of 3,817,031: 6,268,906 pairs, 90s
US rows 3,250,000-3,500,000 of 3,817,031: 6,271,558 pairs, 90s
US rows 3,500,000-3,750,000 of 3,817,031: 6,274,007 pairs, 90s
US rows 3,750,000-3,817,031 of 3